# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)

d

# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:
from pyspark.sql.functions import *

# Extract
bookings_df = spark.table("jarvis_training.default.bookings")

# Transform
result_df = (bookings_df
    .withColumn("starttime_ts", to_timestamp("starttime"))
    .filter(
        (year("starttime_ts") == 2012) &
        (month("starttime_ts") == 9)
    )
    .groupBy("facid")
    .agg(sum("slots").alias("slots"))
    .orderBy("slots"))
# Show result
display(result_df.limit(5))

# Load 
result_df.write.mode("overwrite").parquet("/Volumes/jarvis_training/default/etl_output/job1_parquet")

facid,slots
5,122
3,422
7,426
8,471
6,540


## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
# Extract
members_df = spark.table("jarvis_training.default.members")
facilities_df = spark.table("jarvis_training.default.facilities")

# Transform
result_df = (bookings_df.alias("b")
    .join(members_df.alias("m"), col("b.memid") == col("m.memid"), "inner")
    .join(facilities_df.alias("f"), col("b.facid") == col("f.facid"), "inner")
    .filter(col("f.name").like("Tennis Court%"))
    .select(
        col("f.name").alias("facility"),
        concat_ws(" ", col("m.firstname"), col("m.surname")).alias("member_name"))
    .distinct()
    .orderBy("member_name", "facility")
)

display(result_df.limit(5))

# Load
result_df.write.format("delta").mode("overwrite").partitionBy("facility").saveAsTable("jarvis_training.default.threejoin_delta")

facility,member_name
Tennis Court 1,Anne Baker
Tennis Court 2,Anne Baker
Tennis Court 1,Burton Tracy
Tennis Court 2,Burton Tracy
Tennis Court 1,Charles Owen


## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
import requests
import time
from functools import reduce
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# Alpha Vantage config
API_KEY = "CIE8QQ3CKPFNIZT3"
BASE_URL = "https://www.alphavantage.co/query"

# company -> symbol
symbols = {
    "Google": "GOOGL",
    "Apple": "AAPL",
    "Microsoft": "MSFT",
    "Tesla": "TSLA"
}

# schema for each daily record
schema = StructType([
    StructField("company", StringType(), True),
    StructField("symbol", StringType(), True),
    StructField("trade_date", StringType(), True),
    StructField("open", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("volume", DoubleType(), True)
])

df_list = []

# Extract
for company, symbol in symbols.items():
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "apikey": API_KEY,
        "outputsize": "compact"
    }

    response = requests.get(BASE_URL, params=params, timeout=30)
    data = response.json()

    print(company, symbol)
    print(data.keys())
    print("----------")

    if "Time Series (Daily)" not in data:
        print(f"Failed for {company}: {data}")
        continue

    rows = []
    for trade_date, values in data["Time Series (Daily)"].items():
        rows.append((
            company,
            symbol,
            trade_date,
            float(values["1. open"]),
            float(values["2. high"]),
            float(values["3. low"]),
            float(values["4. close"]),
            float(values["5. volume"])
        ))

    if rows:
        company_df = spark.createDataFrame(rows, schema)
        df_list.append(company_df)

    time.sleep(1)

# Union
if len(df_list) == 0:
    raise ValueError("No stock data retrieved from API.")
elif len(df_list) == 1:
    stock_df = df_list[0]
else:
    stock_df = reduce(lambda df1, df2: df1.union(df2), df_list)

#  Transform
weekly_max_df = (
    stock_df
    .withColumn("trade_date", to_date("trade_date"))
    .withColumn("week", weekofyear("trade_date"))
    .withColumn("year", year("trade_date"))
    .groupBy("company", "year", "week")
    .agg(max("close").alias("max_closing_price"))
    .orderBy("company", "year", "week")
)

display(weekly_max_df.limit(10))

# Load
weekly_max_df.write.format("delta").mode("overwrite").partitionBy("company") .saveAsTable("jarvis_training.default.max_closing_price_weekly")

Google GOOGL
dict_keys(['Meta Data', 'Time Series (Daily)'])
----------
Apple AAPL
dict_keys(['Meta Data', 'Time Series (Daily)'])
----------
Microsoft MSFT
dict_keys(['Meta Data', 'Time Series (Daily)'])
----------
Tesla TSLA
dict_keys(['Meta Data', 'Time Series (Daily)'])
----------


company,year,week,max_closing_price
Apple,2025,1,273.76
Apple,2025,47,271.49
Apple,2025,48,278.85
Apple,2025,49,286.19
Apple,2025,50,278.78
Apple,2025,51,274.61
Apple,2025,52,273.81
Apple,2026,1,271.01
Apple,2026,2,267.26
Apple,2026,3,261.05


## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
# Set up the configuration for the JDBC connection
jdbc_url = "jdbc:postgresql://hh-pgsql-public.ebi.ac.uk:5432/pfmegrnargs"

connection_properties = {
    "user": "reader",
    "password": "NWDMCE5xdipIjRrp",
    "driver": "org.postgresql.Driver"
}

query = "(SELECT * FROM rna LIMIT 100) AS rna_subquery"

# Extract
rna_df = spark.read.jdbc(
    url=jdbc_url,
    table=query,
    properties=connection_properties
)

# Transform
# no transformation required

display(rna_df.limit(5))

# Load
rna_df.write.format("delta").mode("overwrite").saveAsTable("jarvis_training.default.rna_100_records")

id,upi,timestamp,userstamp,crc64,len,seq_short,seq_long,md5
16293310,URS0000F89DBE,2019-12-02T13:19:27.359Z,rnacen,DF55DAF98BAE3649,253,TACGGAGGATGCAAGCGTTATCCGGAATGATTGGGCGTAAAGCGTCCGCAGGTGGCTGTGTAAGTCTGCTGTTAAAGAGTGAGGCTCAACCTCATAAAAGCAGTGGAAACTACACAGCTAGAGTGCGTTCGGGGCAGAGGGAATTCCTGGTGTAGCGGTGAAATGCGTAGAGATCAGGAAGAACACCGGTGGCGAAAGCGTTCTGCTAGACCTGTACTGACACTGAGGGACGAAAGCTAGGGGAGCGAATGGG,null,0d4f27476cb980c23cec7a11a6151643
16293311,URS0000F89DBF,2019-12-02T13:19:27.359Z,rnacen,3E4DC14D5F5274A3,253,TACGAAGGGGGCTAGCGTTGCTCGGAATCACTGGGCGTAAAGGGTGCGTAGGCGGGTCTTTAAGTCAGGGGTGAAATCCTGGAGCTCAACTCCAGAACTGCCTTTGATACTGAAGATCTTGAGTTCGGGAGAGGTGAGTGGAACTGCGAGTGTAGAGGTGAAATTCGTAGATATTCGCAAGAACACCAGTGGCGAAGGCGGCTCACTGGCCCGATACTGACGCTGAGGCGCGAAGGCGTGGGGAGCGAACGGG,null,0d4f2cc3679298b822dcad8a50d794e4
16293312,URS0000F89DC0,2019-12-02T13:19:27.359Z,rnacen,F0C41137905993D1,396,ATACGTAGGTGGCAAGCGTTGTCCGGAATTATTGGGCGTAAAGCGCATGTAGGCGGTGCCTTAAGTCTGTCGTGAAACTGCGGGGCCTAACCCCGTATGGCGATGGAAACTGTGGCCCTTGAGTGCAGGAGAGGAAAGGGGAACTCCCAGTGTAGCGGTTAAATGCGTAGATATTGGGAAGAACACCGGTGGCGAAGGCGCGTTTCTGGACTGTGACTGACGCTGAGATGCGAAAGCCAGGGTAGCGAACGGGATTAGATACCCCGGTAGTCCTGGCCGTAAACGATGGGTACTAGGTGTGGGAGGTATCGACCCCTTCCGTGCCGGAGTTAACGCAATAAGTACCCCACCTGGGGAGTACGGCCGCAAGGCTTAAACTTAAAGGAATTGACGGGG,null,0d4f2e356e09639f05996107935ebb26
16293313,URS0000F89DC1,2019-12-02T13:19:27.359Z,rnacen,12E9361C36A39D12,469,AGGGTTTGATTCTGGCTCAGAACGAACGCTGGCGGCATGCCTAACACATGCAAGTCGAACGAAGGCTTCGGCCTTAGTGGCGCACGAGTGCGTAACGCGTGGGAATCTGCACTTGGGTTCGGAATAACAGCGGGAAACTGCTGCTAATACCGGATGAAGACGAAAGTCCAAAGATTTATCGCCTGAGGATGAGCCCGCGTTGGATTAGGTAGTTGGTGGGGTAAAGGCCTACCAAGCCGACGATCCATAGCTGGTCTGAGAGGATGATCAGCCACACTGGGACTGAGACACGGCCCAGACTCCTACGGGAGGCAGCAGTGGGGAATATTGGACAATGGGCGAAAGCCTGATCCAGCAATGCCGCGTGAGTGATGAAGTCCTTAGGTTTGTAAAGCTCTTTTACCCGGGATGATAATGACAGTACCGGGAGAATAAGCCCCGGCTAACTCCGGGCCAGCAGCCGCGGTAA,null,0d4f33b676c03b9dfd371ecf5196728e
16293314,URS0000F89DC2,2019-12-02T13:19:27.359Z,rnacen,6C593B7B442A5DE8,253,TACAGAGACTGCAAGCGTTATTCGGATTCACTGGGCGTAAAGGGTGCGCAGGCGGCCAAGTGTGTGAGGCGTGAAAGCCCGGGGCTTAACCCCGGAATTGCACCTCAAACTACTTGGCTAGAGCATTGGAGAGGGTAGCAGAATTCACGGTGTGGCAGTGAAATGCGTAGATATCGTGAGGAATACCAGAGGCGAAGGCGGCTACCTGGACAATTGCTGACGCTCAGGCACGAAAGCGTGGGGAGCAAAAGGG,null,0d4f34900f97c9fb19d874d2151d7889
